# Modélisation – H7 : prévoir le prix d'une vente plutôt que la médiane d'un territoire
**Contributeurs :** Cécile · **Date :** 07/11/2026, mis à jour le 10/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre : la table, le masque des lignes, les fenêtres, la naïve, et pour les ventes `evaluer_ventes`, `tester_ventes` (évaluation B) et `evaluer_panier` (évaluation C).

**Question :** un modèle par vente sépare-t-il l'effet du bien et celui du marché, et fait-il mieux que les modèles agrégés à information égale ? Il ne devrait pas mieux anticiper le retournement. Les hypothèses du niveau 2 (volume et élan, capacité d'achat) et les trimestres précédents sont ensuite ajoutés au réseau, pour voir s'ils aident à l'anticiper.

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux.
- `ANNEE_DEBUT_H7` : première année d'apprentissage. Le modèle par vente apprend sur la période longue (à partir de 2017), pour voir plusieurs phases du marché.
- Les blocs de variables sont fixés après le constat rapide (section 2).

In [1]:
HYPOTHESE = "H7"
ANNEE_DEBUT_H7 = 2017      # période longue : apprentissage à partir des ventes de 2017

## 1. Import et chargement
- `charger_table()` : la table de modélisation, avec le masque commun ; elle donne à chaque vente les variables de son sous-bassin (prix N-1, bassin de vie, variables de territoire).
- `charger_ventes(table)` : les ventes de l'échantillon (2014-2024), gardées seulement dans les sous-bassins de la table pour leur année (au moins 20 ventes en N et en N-1). Le prix médian de N et le nombre de ventes de N ne sont pas ajoutés : ils sont inconnus au moment de prévoir. La cible d'une vente est `prix_m2`.
- `charger_ventes_brutes()` : les mêmes ventes sans rattachement, pour le panier fixe (évaluation C).
- `fenetres(..., annee_debut=2017)` : une fenêtre par année jugée (2023, 2024), apprentissage de 2017 à l'année d'avant. L'échantillon 2025 n'est pas chargé.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

import commun
from commun import CIBLE, charger_table, fenetres, charger_ventes, charger_ventes_brutes

pd.set_option("display.width", 200)

table = charger_table()
ventes_brutes = charger_ventes_brutes()
ventes = charger_ventes(table)
FENETRES_TABLE = fenetres(table, annee_debut=ANNEE_DEBUT_H7)     # lignes agrégées (évaluation C)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)          # ventes (apprentissage, évaluation B)

# Ventes gardées par année : celles des sous-bassins sous le seuil ou à case vide sont retirées
garde = pd.DataFrame({"ventes de l'échantillon": ventes_brutes.groupby("anneemut").size(),
                      "ventes rattachées": ventes.groupby("annee").size()})
garde["part gardée (%)"] = (garde["ventes rattachées"] / garde["ventes de l'échantillon"] * 100).round(1)
print(garde.loc[ANNEE_DEBUT_H7:].to_string())

# Contrôles : prix N-1 et bassin de vie connus pour chaque vente
assert ventes["prix_n1"].notna().all() and ventes["BV2022"].notna().all()
assert ventes["annee"].max() <= commun.ANNEES_JUGEES[-1], "aucune vente de 2025 ne doit être chargée"

for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(int(x) for x in app['annee'].unique())} "
          f"({len(app)} ventes) | jugée {a} ({len(ev)} ventes)")

      ventes de l'échantillon  ventes rattachées  part gardée (%)
2017                    83523            82725.0             99.0
2018                    83304            82555.0             99.1
2019                    91533            90756.0             99.2
2020                    89204            88529.0             99.2
2021                   102718           101956.0             99.3
2022                    98340            97692.0             99.3
2023                    78477            77828.0             99.2
2024                    71808            71127.0             99.1
Fenêtre 2023 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022] (544213 ventes) | jugée 2023 (77828 ventes)
Fenêtre 2024 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022, 2023] (622041 ventes) | jugée 2024 (71127 ventes)


## 2. Constat rapide : les variables du bien
Avant de choisir les transformations, on regarde les variables candidates du bien sur les années d'apprentissage de la première fenêtre (2017-2022) : les années jugées ne sont pas regardées. Par type de bien : cases vides, valeurs nulles, distribution, nombre de pièces, et cohérence entre surface et nombre de pièces.

Variables candidates : surface bâtie (`sbati`), surface du terrain (`sterr`), nombre de pièces (`nb_pieces`, 5 = 5 ou plus), nombre de dépendances (`nblocdep`), mois de la vente (`moismut`), coordonnées de la parcelle (`geompar_x`, `geompar_y`).

In [3]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
BIEN_CANDIDATES = ["sbati", "sterr", "nb_pieces", "nblocdep", "moismut", "geompar_x", "geompar_y"]
par_type = app_constat.groupby("type_bien")

print("Types :", app_constat[BIEN_CANDIDATES].dtypes.astype(str).to_dict())

# Cases vides et valeurs nulles (%), par type de bien
print("\nCases vides (%)")
print(par_type[BIEN_CANDIDATES].apply(lambda g: g.isna().mean() * 100).T.round(2).to_string())
print("\nValeurs à 0 (%)")
print(par_type[BIEN_CANDIDATES].apply(lambda g: (g == 0).mean() * 100).T.round(2).to_string())

# Distribution, par type de bien
for type_bien, e in par_type:
    print(f"\n{type_bien} ({len(e)} ventes)")
    print(e[BIEN_CANDIDATES].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(1).to_string())

# Répartition du nombre de pièces (%) et surface médiane par nombre de pièces
print("\nNombre de pièces (% des ventes de chaque type)")
print(pd.crosstab(app_constat["nb_pieces"], app_constat["type_bien"], normalize="columns").mul(100).round(1).to_string())
print("\nSurface bâtie médiane (m²) par nombre de pièces")
print(app_constat.pivot_table(index="nb_pieces", columns="type_bien", values="sbati", aggfunc="median").to_string())

Types : {'sbati': 'float64', 'sterr': 'float64', 'nb_pieces': 'int64', 'nblocdep': 'int64', 'moismut': 'int64', 'geompar_x': 'float64', 'geompar_y': 'float64'}

Cases vides (%)
type_bien  appartement  maison
sbati             0.00    0.00
sterr             0.00    0.00
nb_pieces         0.00    0.00
nblocdep          0.00    0.00
moismut           0.00    0.00
geompar_x         0.01    0.02
geompar_y         0.01    0.02

Valeurs à 0 (%)
type_bien  appartement  maison
sbati             0.00    0.00
sterr            98.33    4.41
nb_pieces         0.00    0.00
nblocdep         38.52   83.88
moismut           0.00    0.00
geompar_x         0.00    0.00
geompar_y         0.00    0.00

appartement (228029 ventes)
              count       mean       std        min         1%         5%        50%        95%        99%         max
sbati      228029.0       58.3      27.8       10.0       15.0       21.0       56.0      104.0      147.0       735.0
sterr      228029.0       12.7     663.5   

### 2.1 Préparation des variables du bien
Transformations décidées d'après le constat :
- **Surface bâtie** en log : un mètre carré de plus ne compte pas pareil dans un studio et dans une grande maison.
- **Terrain** : mis à 0 pour les appartements (la surface relevée est celle de la copropriété, pas du logement) ; pour les maisons, borné au 99e centile de 2017-2022, pour que les quelques ventes avec des terres agricoles ne pèsent pas sur le modèle, puis en log(1 + x) (beaucoup de petites valeurs et des zéros).
- **Nombre de pièces** : une indicatrice par valeur (1, 2, 3, 4, 5 ou plus).
- **Dépendances** : une indicatrice par catégorie (0, 1, 2, 3 ou plus).
- Le **mois de la vente** n'est pas retenu : on ne cherche pas la saisonnalité.
- Les **coordonnées de la parcelle** ne font pas partie du bloc du bien. Elles forment un bloc à part, testé en H7v1b : l'hypothèse est qu'elles sont redondantes avec le sous-bassin. Les coordonnées hors de la France métropolitaine et les cases vides (moins de 0,03 % des ventes) sont remplacées par la médiane des coordonnées du sous-bassin sur 2017-2022.

Les colonnes indicatrices sont écrites une à une, pour être identiques dans tous les jeux (apprentissage, année jugée, panier fixe), même si une catégorie y est absente. Les mêmes transformations sont appliquées aux ventes brutes, qui servent au panier fixe.

In [4]:
# Seuil du terrain des maisons : 99e centile de 2017-2022 (années d'apprentissage de la première fenêtre)
SEUIL_STERR = app_constat.loc[app_constat["type_bien"] == "maison", "sterr"].quantile(0.99)

# Coordonnées (Lambert 93) : limites de la France métropolitaine, Corse comprise
BORNES_X, BORNES_Y = (100_000, 1_250_000), (6_030_000, 7_120_000)

COLONNES_PIECES = ["pieces_1", "pieces_2", "pieces_3", "pieces_4", "pieces_5_plus"]
COLONNES_DEP = ["dep_0", "dep_1", "dep_2", "dep_3_plus"]
BLOC_BIEN = ["maison", "log_sbati", "log1p_sterr"] + COLONNES_PIECES + COLONNES_DEP
BLOC_COORD = ["geompar_x", "geompar_y"]


def coordonnees_douteuses(v):
    """Vrai pour une coordonnée vide ou hors de la France métropolitaine."""
    return (~v["geompar_x"].between(*BORNES_X)) | (~v["geompar_y"].between(*BORNES_Y))


# Médiane des coordonnées de chaque sous-bassin, sur les coordonnées valides de 2017-2022
valides = app_constat[~coordonnees_douteuses(app_constat)]
CENTRES = valides.groupby("sous_bassin_de_vie")[BLOC_COORD].median()


def preparer_bien(v):
    """Variables du bien et coordonnées : transformations ligne à ligne, seuils fixés d'après 2017-2022."""
    v = v.copy()
    v["log_sbati"] = np.log(v["sbati"])
    terrain = v["sterr"].where(v["type_bien"] == "maison", 0).clip(upper=SEUIL_STERR)
    v["log1p_sterr"] = np.log1p(terrain)
    for k, col in enumerate(COLONNES_PIECES, start=1):          # 5 = 5 pièces ou plus
        v[col] = (v["nb_pieces"] == k).astype(int)
    dependances = v["nblocdep"].clip(upper=3)                   # 3 = 3 dépendances ou plus
    for k, col in enumerate(COLONNES_DEP):
        v[col] = (dependances == k).astype(int)
    douteuses = coordonnees_douteuses(v)
    for c in BLOC_COORD:
        v.loc[douteuses, c] = v.loc[douteuses, "sous_bassin_de_vie"].map(CENTRES[c])
    return v


print("Coordonnées remplacées (ventes 2017-2024) :", int(coordonnees_douteuses(ventes).sum()))
ventes = preparer_bien(ventes)
ventes_brutes = preparer_bien(ventes_brutes)                    # pour le panier fixe (évaluation C)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)        # fenêtres refaites avec les nouvelles colonnes

# Contrôles
assert ventes[BLOC_BIEN + BLOC_COORD].notna().all().all(), "case vide dans le bloc du bien ou les coordonnées"
assert not coordonnees_douteuses(ventes).any(), "coordonnée encore hors de la France métropolitaine"
assert (ventes[COLONNES_PIECES].sum(axis=1) == 1).all() and (ventes[COLONNES_DEP].sum(axis=1) == 1).all(), \
    "chaque vente doit avoir exactement une catégorie de pièces et une de dépendances"
assert (ventes.loc[ventes["type_bien"] == "appartement", "log1p_sterr"] == 0).all(), "terrain des appartements non nul"

maisons = app_constat["type_bien"] == "maison"
print(f"Seuil du terrain des maisons : {SEUIL_STERR:,.0f} m² | maisons bornées en 2017-2022 : "
      f"{(app_constat.loc[maisons, 'sterr'] > SEUIL_STERR).mean() * 100:.1f} %")
app_bien = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_bien.groupby("type_bien")[BLOC_BIEN].mean().T.round(3).to_string())

Coordonnées remplacées (ventes 2017-2024) : 152
Seuil du terrain des maisons : 26,704 m² | maisons bornées en 2017-2022 : 1.0 %
type_bien      appartement  maison
maison               0.000   1.000
log_sbati            3.953   4.546
log1p_sterr          0.000   6.284
pieces_1             0.166   0.021
pieces_2             0.297   0.064
pieces_3             0.312   0.192
pieces_4             0.172   0.330
pieces_5_plus        0.054   0.393
dep_0                0.385   0.839
dep_1                0.426   0.118
dep_2                0.152   0.034
dep_3_plus           0.037   0.009


### 2.2 Lien avec le prix, redondances
Sur les années d'apprentissage de la première fenêtre (2017-2022), par type de bien, rho de Spearman entre chaque variable du bien et :
- le **prix au m²** de la vente : ce lien mélange l'effet du bien et celui du lieu (les petits logements sont plus fréquents dans les villes chères) ;
- l'**écart au prix médian N-1 du sous-bassin** (prix au m² / prix N-1 − 1) : le niveau du lieu est en grande partie retiré, il reste ce que le bien explique. C'est ce que le modèle par vente doit apprendre en plus du sous-bassin.

Redondances : couples de variables du bloc dont |rho| dépasse 0,7. Avec plusieurs centaines de milliers de ventes, tout rho non nul serait significatif : les rho décrivent la force des liens, ils ne se testent pas.

In [5]:
ecart_n1 = app_bien["prix_m2"] / app_bien["prix_n1"] - 1
VARIABLES_LIEN = [v for v in BLOC_BIEN if v != "maison"]       # maison est constante dans chaque type de bien

liens = []
for type_bien, e in app_bien.groupby("type_bien"):
    for v in VARIABLES_LIEN:
        if e[v].nunique() < 2:                                  # ex. terrain des appartements, toujours à 0
            continue
        liens.append({"type de bien": type_bien, "variable": v,
                      "rho avec le prix au m²": stats.spearmanr(e[v], e["prix_m2"]).statistic,
                      "rho avec l'écart au prix N-1": stats.spearmanr(e[v], ecart_n1.loc[e.index]).statistic})
liens = pd.DataFrame(liens).pivot(index="variable", columns="type de bien")
print(liens.reindex(VARIABLES_LIEN).round(2).to_string())

couples = []
for type_bien, e in app_bien.groupby("type_bien"):
    rho = e[VARIABLES_LIEN].corr(method="spearman")
    for i, a in enumerate(VARIABLES_LIEN):
        for b in VARIABLES_LIEN[i + 1:]:
            if abs(rho.loc[a, b]) > 0.7:
                couples.append({"type de bien": type_bien, "variable 1": a, "variable 2": b,
                                "rho": round(rho.loc[a, b], 2)})
print(pd.DataFrame(couples).to_string(index=False) if couples else "Aucun couple au-delà de 0,7")

              rho avec le prix au m²        rho avec l'écart au prix N-1       
type de bien             appartement maison                  appartement maison
variable                                                                       
log_sbati                      -0.21  -0.04                        -0.18  -0.11
log1p_sterr                      NaN  -0.10                          NaN   0.15
pieces_1                        0.14  -0.04                         0.13  -0.02
pieces_2                        0.08  -0.04                         0.08   0.00
pieces_3                       -0.06  -0.03                        -0.05   0.02
pieces_4                       -0.13   0.05                        -0.13   0.05
pieces_5_plus                  -0.05   0.00                        -0.06  -0.06
dep_0                           0.02  -0.06                        -0.04  -0.06
dep_1                          -0.02   0.06                         0.00   0.04
dep_2                          -0.00   0

**Comment lire :** le rho de Spearman va de −1 à +1 ; son signe donne le sens du lien, sa valeur absolue la force (environ 0,1 : faible ; 0,3 : moyen ; 0,5 et plus : fort). Exemple : pour les appartements, le rho entre `log_sbati` et le prix au m² vaut −0,21 : plus le logement est grand, plus le m² est bon marché, avec un lien faible à moyen. Il vaut encore −0,18 avec l'écart au prix N-1 : l'effet tient à l'intérieur d'un même sous-bassin, c'est un effet du bien et non du lieu.

**Constat :**
- Les liens entre les variables du bien et le prix sont faibles : le plus fort est celui de la surface des appartements (−0,21 avec le prix au m², −0,18 avec l'écart au prix N-1). Le nombre de pièces des appartements suit la surface : les studios ont un m² plus cher (+0,13), les 4 pièces un m² moins cher (−0,13). Les dépendances n'ont presque aucun lien avec le prix (|rho| ≤ 0,06).
- Pour les maisons, le terrain change de signe : −0,10 avec le prix au m², +0,15 avec l'écart au prix N-1. Les grands terrains se trouvent surtout là où les prix sont bas, mais dans un même sous-bassin, plus de terrain donne un prix au m² plus élevé : le prix de vente comprend le terrain et il est divisé par la seule surface bâtie. La surface bâtie des maisons montre le même effet, plus faible (−0,04, puis −0,11).
- Seul couple au-delà de 0,7 : `dep_0` et `dep_1` pour les maisons (−0,84), un effet mécanique de l'encodage (84 % des maisons sans dépendance, 12 % avec une seule). Les deux variables sont gardées.
- Ces rho ne mesurent que des liens monotones, une variable à la fois ; le réseau peut apprendre des combinaisons (surface et lieu). On attend un gain modeste du bien par rapport au sous-bassin seul.

## 3. Fonctions du notebook
Chaque modèle est lancé en une ligne : `lancer(nom, famille, variables, **réglages)`.
- `famille` : `"keras"` (gabarit `gabarit_keras.py`) ou `"elastic"` (Elastic Net par vente, ci-dessous) ;
- `variables` : les colonnes données au modèle, dont `sous_bassin_de_vie` quand le lieu est utilisé ;
- un seul modèle pour les deux types de bien (`maison` est une variable), poids `w_h` dans l'apprentissage et dans les mesures.

Pour chaque année jugée (2023, puis 2024), `lancer` :
1. apprend sur les ventes de 2017 à l'année d'avant ;
2. prévoit le prix au m² de chaque vente de l'année jugée : **évaluation B** (`evaluer_ventes`, `tester_ventes`), naïve = prix médian N-1 du sous-bassin de la vente ;
3. prévoit, pour l'année jugée, le prix des biens vendus l'année d'avant (panier fixe) : la médiane pondérée de ces prévisions est comparée au prix médian réel du sous-bassin, comme pour les modèles agrégés : **évaluation C** (`evaluer_panier`).

**Elastic Net par vente.** Le modèle pont (H4) est un Elastic Net appris sur la table agrégée : son gabarit n'est pas modifié, la même méthode est adaptée ici aux ventes.
- Il reçoit exactement les mêmes colonnes que la version du réseau à laquelle il est comparé.
- La cible est le log du prix au m², remise en €/m² à la prévision ; les variables sont mises à l'échelle (moyennes et écarts-types calculés sur les ventes d'apprentissage).
- Un modèle linéaire ne peut pas utiliser directement 1 700 sous-bassins comme catégories. Chaque sous-bassin est remplacé par un code appris : la moyenne du log du prix de ses ventes d'apprentissage (`TargetEncoder`). Ce code est calculé avec un découpage interne en 5 parts : une vente ne sert jamais à calculer son propre code. Un sous-bassin inconnu reçoit la moyenne générale.
- La grille du modèle pont (`alpha` et `l1_ratio`) est évaluée sur la dernière année d'apprentissage, comme l'arrêt du réseau ; le meilleur réglage est ensuite réappris sur toute la fenêtre. Pas de validation croisée sur tous les plis : avec plus de 500 000 ventes, le temps de calcul serait trop long.

In [6]:
import time
import warnings
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, TargetEncoder

from commun import evaluer_ventes, tester_ventes, panier_fixe, evaluer_panier
from gabarit_pont import GRILLE_EN
from gabarit_keras import prevoir_keras

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # alpha très petits : convergence lente, sans effet sur la prévision
resultats, tests, ajustes, previsions = [], [], {}, {}


class ElasticNetVentes(RegressorMixin, BaseEstimator):
    """Elastic Net par vente : apprend log(prix au m²), rend des €/m².

    Mêmes colonnes que la version Keras comparée (variables déjà transformées en log dans le notebook).
    Le sous-bassin est remplacé par un code appris (TargetEncoder, découpage interne en 5 parts).
    La colonne 'annee', si elle est présente, n'est jamais une variable.
    """

    def __init__(self, colonne_lieu="sous_bassin_de_vie", alpha=1e-3, l1_ratio=0.5, random_state=commun.GRAINE):
        self.colonne_lieu = colonne_lieu
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.random_state = random_state

    def _variables(self, X, code):
        Xn = X[self.variables_].astype(float)      # mêmes colonnes qu'à l'apprentissage, dans le même ordre
        return Xn if code is None else Xn.assign(code_lieu=code)

    def fit(self, X, y, sample_weight=None):
        cible = np.log(np.asarray(y, dtype=float))
        self.avec_lieu_ = self.colonne_lieu in X.columns
        self.variables_ = [c for c in X.columns if c not in ("annee", self.colonne_lieu)]
        code = None
        if self.avec_lieu_:
            self.encodeur_ = TargetEncoder(target_type="continuous",
                                           cv=KFold(5, shuffle=True, random_state=self.random_state))
            code = self.encodeur_.fit_transform(X[[self.colonne_lieu]], cible).ravel()
        Xn = self._variables(X, code)
        self.echelle_ = StandardScaler().fit(Xn, sample_weight=sample_weight)
        # precompute=True : les liens entre variables sont calculés une seule fois (petit tableau variables x variables) ;
        # chaque tour de l'ajustement ne relit plus les centaines de milliers de ventes. Mêmes coefficients, beaucoup plus rapide
        self.modele_ = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, max_iter=50_000, precompute=True)
        self.modele_.fit(self.echelle_.transform(Xn), cible, sample_weight=sample_weight)
        self.coefs_ = pd.Series(self.modele_.coef_, index=Xn.columns)     # coefficients standardisés
        return self

    def predict(self, X):
        code = self.encodeur_.transform(X[[self.colonne_lieu]]).ravel() if self.avec_lieu_ else None
        Z = self.echelle_.transform(self._variables(X, code))
        return pd.Series(np.exp(self.modele_.predict(Z)), index=X.index)


def prevoir_elastic_ventes(app, ev, variables, grille=None):
    """Évalue la grille sur la dernière année d'apprentissage, réapprend sur toute la fenêtre, prévoit ev."""
    grille = GRILLE_EN if grille is None else grille
    colonnes = list(variables) + ["annee"]
    derniere = app["annee"].max()
    ent, arr = app[app["annee"] < derniere], app[app["annee"] == derniere]
    essais = []
    for reglage in ParameterGrid(grille):
        m = ElasticNetVentes(**reglage).fit(ent[colonnes], ent["prix_m2"], sample_weight=ent["w_h"])
        mae = np.average((arr["prix_m2"] - m.predict(arr[colonnes])).abs(), weights=arr["w_h"])
        essais.append({**reglage, "MAE année d'arrêt (€/m²)": mae})
    essais = pd.DataFrame(essais).sort_values("MAE année d'arrêt (€/m²)")
    meilleur = {k: essais.iloc[0][k] for k in grille}
    modele = ElasticNetVentes(**meilleur).fit(app[colonnes], app["prix_m2"], sample_weight=app["w_h"])
    modele.essais_ = essais
    return modele.predict(ev[colonnes]), modele


def lancer(nom, famille, variables, **reglages):
    """Un modèle par fenêtre (un seul pour les deux types de bien), puis évaluations B et C.

    Réseau Keras : les réglages optimisés (REGLAGES_KERAS, section 3.1) s'appliquent à toutes les versions ;
    un réglage passé à lancer garde la priorité (ex. colonne_embedding=None). Si la cible retenue est
    l'évolution et que la version a le prix N-1, le réseau apprend le rapport prix / prix N-1.
    """
    assert famille in ("keras", "elastic")
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if not r["modèle"].startswith(nom + " (")]
    tests[:] = [t for t in tests if not t["modèle"].startswith(nom + " (")]
    if famille == "keras":
        reglages = {**REGLAGES_KERAS, **reglages}
        if EVOLUTION and "log_prix_n1" in variables:
            reglages.setdefault("reference", "log_prix_n1")
    for annee, (app, ev) in FENETRES.items():
        debut = time.time()
        panier = panier_fixe(ventes_brutes, table, annee)        # biens vendus en N-1, datés de N
        if famille == "keras":
            prevision, m = prevoir_keras(app, ev, variables, "prix_m2", poids="w_h", **reglages)
        else:
            prevision, m = prevoir_elastic_ventes(app, ev, variables, **reglages)
        prevision_panier = m.predict(panier)
        ajustes[(nom, annee)] = m
        previsions[(nom, annee)] = (prevision, prevision_panier)
        evaluer_ventes(nom, ev, prevision, resultats)
        tester_ventes(nom, ev, prevision, tests)
        evaluer_panier(nom, panier, prevision_panier, table, resultats, tests)
        print(f"{nom} | fenêtre {annee} : {(time.time() - debut) / 60:.1f} min")

    res = pd.DataFrame(resultats)
    res = res[res["modèle"].str.startswith(nom + " (")][["modèle", "type de bien", "année jugée", "MAE (€/m²)",
                                                         "MAE naïve (€/m²)", "gain sur la naïve (%)",
                                                         "bassins où le modèle gagne (%)"]]
    res["évaluation"] = res.pop("modèle").str.extract(r"\((B|C)", expand=False)
    print(nom, f"({len(variables)} variables)", reglages if famille == "keras" else "")
    print(res.round(1).to_string(index=False))
    return nom

I0000 00:00:1794408020.523604    5071 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794408020.524256    5071 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794408020.602316    5071 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794408021.831037    5071 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

### 3.1 Réglages du réseau
Les réglages des réseaux Keras viennent de `optimisation_keras.py` (fichier `meilleurs_reglages_keras.json`). Ils ont été choisis sur une année de validation (2022) qui n'est jamais une année jugée, avec des ventes de 2017 à 2021 seulement : aucune année jugée n'a servi au choix.
- **Cible** : quand la version dispose du prix N-1, le réseau apprend le rapport entre le prix de la vente et le prix médian N-1 de son sous-bassin, puis prévoit prix N-1 × rapport. Le niveau de chaque territoire vient ainsi du prix N-1 : le réseau ne ramène plus les prix élevés vers le milieu (appartements les plus chers). Les versions sans prix N-1 (H7v1 à H7v2b) apprennent toujours le prix.
- **Perte** : erreur absolue (MAE), la même mesure que celle qui juge les modèles.
- **Réglages** : couches, dropout, taux d'apprentissage et taille de l'embedding retenus par une grille complète.
- Un seul réseau pour les deux types de bien : un réseau par type a été testé et n'apporte rien de stable.

In [ ]:
import json

with open(commun.DATA_OUT / "meilleurs_reglages_keras.json", encoding="utf-8") as f:
    OPTIMISATION_KERAS = json.load(f)["reglages"]

assert OPTIMISATION_KERAS["organisation"] == "commun", "H7 n'utilise qu'un réseau pour les deux types de bien"
EVOLUTION = OPTIMISATION_KERAS["cible"] == "evolution"
REGLAGES_KERAS = {k: OPTIMISATION_KERAS[k] for k in ["dropout", "learning_rate", "dim_embedding", "perte"]}
REGLAGES_KERAS["couches"] = tuple(OPTIMISATION_KERAS["couches"])

print("Cible :", "évolution par rapport au prix N-1" if EVOLUTION else "prix", "| réglages :", REGLAGES_KERAS)

## 4. Modèles
D'abord le « où », puis le « quand ». Chaque version Keras est comparée à un Elastic Net par vente qui dispose des mêmes informations : l'Elastic Net est le modèle retenu au niveau 1 (H3) et pour le modèle pont (H4). Tous les modèles sont gardés, y compris ceux qui perdent.

### 4.1 Référence du « où » : Elastic Net, sous-bassin + bien
Mêmes informations que H7v1 : le lieu (sous-bassin codé) et les variables du bien, sans prix N-1 ni indicateur de la phase du marché.

In [28]:
REFERENCE_OU = lancer("H7 – Elastic Net – sous-bassin + bien", "elastic", ["sous_bassin_de_vie"] + BLOC_BIEN)

# Réglages essayés sur la dernière année d'apprentissage (3 meilleurs) et coefficients standardisés du modèle réappris
for annee in commun.ANNEES_JUGEES:
    m = ajustes[(REFERENCE_OU, annee)]
    print(f"\nFenêtre {annee}")
    print(m.essais_.head(3).round(5).to_string(index=False))
    print(m.coefs_.round(3).to_string())

H7 – Elastic Net – sous-bassin + bien | fenêtre 2023 : 0.2 min
H7 – Elastic Net – sous-bassin + bien | fenêtre 2024 : 0.2 min
H7 – Elastic Net – sous-bassin + bien (13 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       844.8             732.6                  -15.3                            31.2          B
      maison         2023       861.0             774.2                  -11.2                            30.1          B
 appartement         2023       497.9             166.2                 -199.5                            15.2          C
      maison         2023       458.9             111.7                 -310.8                             8.0          C
 appartement         2024       841.1             744.9                  -12.9                            35.8          B
      maison         2024       788.8             740.6                   -6.5           

**Comment lire :** le tableau donne, pour chaque année jugée et chaque type de bien, l'erreur moyenne du modèle (MAE, en €/m²), celle de la naïve et le gain sur la naïve (négatif : le modèle fait moins bien). L'évaluation B juge chaque vente (naïve = prix médian N-1 de son sous-bassin) ; l'évaluation C juge le prix médian de chaque sous-bassin, comme les modèles agrégés. Exemple : en 2023, pour les appartements, l'Elastic Net se trompe en moyenne de 845 €/m² par vente, contre 733 €/m² pour la naïve, soit un gain de −15,3 % ; il ne fait mieux que la naïve que dans 31 % des sous-bassins.

Sous ce tableau : les trois meilleurs réglages essayés sur la dernière année d'apprentissage, puis les coefficients standardisés du modèle réappris (variables centrées-réduites, prix en log : ils se comparent entre eux, pas à des €/m²).

**Résultats :**
- Sans information sur le moment, l'Elastic Net perd contre la naïve partout : de −6,5 à −15,3 % en évaluation B, de −178 à −311 % en évaluation C.
- Le réglage retient un `alpha` très petit (le plus petit de la grille pour la fenêtre 2023) : avec 13 variables et environ 450 000 ventes, la pénalité ne sert presque à rien.
- Après le lieu (`code_lieu`, 0,58), le terrain (+0,16) et la surface (−0,12) pèsent le plus, comme l'annonçait le constat (section 2.2). `pieces_3` et `dep_1` valent exactement 0 : les indicatrices d'un même groupe font toujours 1 au total, et l'Elastic Net en retire une de lui-même.

### 4.2 H7v1 : réseau Keras, sous-bassin (embedding) + bien
Mêmes informations que la référence 4.1 : le lieu, appris par un embedding (un vecteur de 8 valeurs par sous-bassin), et les variables du bien. Réglages du gabarit par défaut.

**Biais des prévisions (évaluation B).** Les modèles du « où » ne savent pas en quelle année ils prévoient : ils apprennent un niveau moyen de la période 2017 à l'année d'avant. Le biais est l'écart moyen entre prévision et prix réel, en % (moyenne des log, pondérée par `w_h`) : négatif, le modèle sous-estime.

In [29]:
H7V1 = lancer("H7v1 – Keras – sous-bassin + bien", "keras", ["sous_bassin_de_vie"] + BLOC_BIEN)


def biais(noms):
    """Écart moyen prévision / prix réel (%) par modèle, année jugée et type de bien (évaluation B)."""
    lignes = []
    for nom in noms:
        for annee, (app, ev) in FENETRES.items():
            ecart = np.log(previsions[(nom, annee)][0] / ev["prix_m2"])
            for type_bien, e in ev.groupby("type_bien"):
                lignes.append({"modèle": nom, "année jugée": annee, "type de bien": type_bien,
                               "biais (%)": (np.exp(np.average(ecart.loc[e.index], weights=e["w_h"])) - 1) * 100})
    return pd.DataFrame(lignes).pivot_table(index=["modèle", "type de bien"], columns="année jugée",
                                            values="biais (%)").round(1)


biais([REFERENCE_OU, H7V1])

H7v1 – Keras – sous-bassin + bien | fenêtre 2023 : 1.4 min
H7v1 – Keras – sous-bassin + bien | fenêtre 2024 : 1.7 min
H7v1 – Keras – sous-bassin + bien (13 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       798.2             732.6                   -8.9                            41.2          B
      maison         2023       836.9             774.2                   -8.1                            39.6          B
 appartement         2023       451.6             166.2                 -171.7                            18.7          C
      maison         2023       443.5             111.7                 -297.0                             9.9          C
 appartement         2024       870.7             744.9                  -16.9                            37.2          B
      maison         2024       759.9             740.6                   -2.6                       

année jugée                                         2023  2024
modèle                                type de bien            
H7 – Elastic Net – sous-bassin + bien appartement   -8.9  -5.6
                                      maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien     appartement   -8.7 -12.5
                                      maison       -12.3  -8.7

In [30]:
for a in commun.ANNEES_JUGEES:
    m = ajustes[(H7V1, a)]
    print(a, "| meilleure époque :", m.meilleure_epoque_, "| époques faites :", len(m.historique_["perte_arret"]),
          "| perte d'arrêt :", [round(p, 3) for p in m.historique_["perte_arret"][:20]])

2023 | meilleure époque : 7 | époques faites : 22 | perte d'arrêt : [0.455, 0.431, 0.43, 0.43, 0.431, 0.428, 0.425, 0.428, 0.427, 0.433, 0.428, 0.434, 0.437, 0.429, 0.427, 0.432, 0.432, 0.432, 0.432, 0.431]
2024 | meilleure époque : 8 | époques faites : 23 | perte d'arrêt : [0.437, 0.431, 0.424, 0.428, 0.427, 0.425, 0.425, 0.423, 0.426, 0.427, 0.427, 0.429, 0.427, 0.426, 0.429, 0.432, 0.428, 0.43, 0.428, 0.43]


**Résultats :**
- H7v1 perd aussi contre la naïve : de −2,6 à −16,9 % en évaluation B, de −173 à −297 % en évaluation C.
- À informations égales, le réseau fait mieux que l'Elastic Net en 2023 (évaluation B : 798 contre 845 €/m² pour les appartements, 837 contre 861 pour les maisons) et pour les maisons en 2024 (760 contre 789) ; l'Elastic Net fait mieux pour les appartements en 2024 (841 contre 871). Les tests de la section 5.2 disent si ces écarts dépassent le hasard.
- Les deux modèles sous-estiment les prix, de 6 à 13 % : sans prix N-1 ni indicateur de la phase du marché, ils ne savent pas en quelle année ils prévoient. Pour un lieu et un bien, ils prévoient un niveau moyen de la période d'apprentissage, alors que les prix ont fortement monté entre 2017 et 2022. Le biais de l'Elastic Net baisse en 2024 (−5,6 % et −6,8 %), après la baisse des prix et avec 2023 dans l'apprentissage ; celui du réseau reste fort pour les appartements (−12,5 %).
- En évaluation B, ce que le modèle gagne en tenant compte du bien est plus que perdu par l'erreur de niveau ; en évaluation C, l'effet du bien disparaît dans la médiane et seule l'erreur de niveau reste.
- Arrêt du réseau (cellule ci-dessus) : la meilleure époque est la 7e ou la 8e, et la perte d'arrêt ne baisse presque plus après la 2e ou la 3e. Ce n'est pas l'entraînement qui limite le modèle, c'est l'information qui manque.

### 4.3 H7v1b : réseau Keras, sous-bassin (embedding) + bien + coordonnées
H7v1 avec, en plus, les coordonnées de la parcelle. Hypothèse : la localisation fine est redondante avec le sous-bassin ; si c'est le cas, H7v1b ne fait pas mieux que H7v1.

In [33]:
H7V1B = lancer("H7v1b – Keras – sous-bassin + bien + coordonnées", "keras",
               ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_COORD)
biais([REFERENCE_OU, H7V1, H7V1B])

H7v1b – Keras – sous-bassin + bien + coordonnées | fenêtre 2023 : 2.8 min
H7v1b – Keras – sous-bassin + bien + coordonnées | fenêtre 2024 : 2.4 min
H7v1b – Keras – sous-bassin + bien + coordonnées (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       792.6             732.6                   -8.2                            42.8          B
      maison         2023       832.3             774.2                   -7.5                            38.8          B
 appartement         2023       451.9             166.2                 -171.9                            17.0          C
      maison         2023       448.0             111.7                 -301.1                             9.3          C
 appartement         2024       866.5             744.9                  -16.3                            37.3          B
      maison         2024       753.0             740.6 

année jugée                                                    2023  2024
modèle                                           type de bien            
H7 – Elastic Net – sous-bassin + bien            appartement   -8.9  -5.6
                                                 maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                appartement   -8.7 -12.5
                                                 maison       -12.3  -8.7
H7v1b – Keras – sous-bassin + bien + coordonnées appartement   -9.0 -12.4
                                                 maison       -13.1  -7.8

**Résultats :** les coordonnées n'apportent presque rien au sous-bassin. En évaluation B, l'erreur de H7v1b ne diffère de celle de H7v1 que de 4 à 7 €/m² selon l'année et le type de bien (moins de 1 %) ; en évaluation C, les résultats sont aussi les mêmes, et le biais reste identique (de −8 à −13 %). Le test (section 5.2) détecte un gain minime pour les maisons, sans portée pratique. Ce qui manque n'est pas une localisation plus fine, mais l'information sur le moment.

### 4.4 Le territoire : variables retenues au niveau 1
Variables de territoire retenues au niveau 1 (H3 globale) : le parc de logements (9 indicateurs), la vacance (LOVAC) et la densité. Ce sont des photos fixes : pour un sous-bassin, elles ont la même valeur chaque année. Les variables asymétriques passent en log (mêmes règles que les autres notebooks), dans la table et dans les ventes : le panier fixe de l'évaluation C reprend ses variables dans la table.

In [7]:
PARC = ["taux_vacance", "part_rsec", "part_apparts", "log_nb_logements", "part_moins40m2",
        "part_avant1946", "part_2006_2019", "anciennete_moyenne", "part_sans_voiture"]
VACANCE = ["taux_vacance_complete"]
DENSITE = ["densite_population"]
LOG_TERRITOIRE = [v for v in PARC + VACANCE + DENSITE
                  if v in ["densite_population", "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts"]]
BLOC_TERRITOIRE = ["log_" + v if v in LOG_TERRITOIRE else v for v in PARC + VACANCE + DENSITE]


def preparer_territoire(t):
    """Variables de territoire asymétriques en log (transformation ligne à ligne)."""
    t = t.copy()
    assert (t[LOG_TERRITOIRE] > 0).all().all(), "valeur nulle ou négative dans une variable passée en log"
    for v in LOG_TERRITOIRE:
        t["log_" + v] = np.log(t[v])
    return t


table = preparer_territoire(table)              # pour le panier fixe (évaluation C)
ventes = preparer_territoire(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)
assert ventes[BLOC_TERRITOIRE].notna().all().all(), "case vide dans le bloc du territoire"
print(len(BLOC_TERRITOIRE), "variables de territoire :", BLOC_TERRITOIRE)

11 variables de territoire : ['taux_vacance', 'log_part_rsec', 'log_part_apparts', 'log_nb_logements', 'log_part_moins40m2', 'part_avant1946', 'part_2006_2019', 'anciennete_moyenne', 'log_part_sans_voiture', 'taux_vacance_complete', 'log_densite_population']


### 4.5 H7v2a : réseau Keras, sous-bassin (embedding) + bien + territoire
H7v1 avec, en plus, les variables de territoire. Question : le territoire ajoute-t-il quelque chose à l'identité du sous-bassin ? L'embedding apprend déjà ce qui est propre et constant pour chaque sous-bassin : on attend un gain faible.

In [35]:
H7V2A = lancer("H7v2a – Keras – sous-bassin + bien + territoire", "keras",
               ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE)
biais([REFERENCE_OU, H7V1, H7V1B, H7V2A])

H7v2a – Keras – sous-bassin + bien + territoire | fenêtre 2023 : 2.2 min
H7v2a – Keras – sous-bassin + bien + territoire | fenêtre 2024 : 1.7 min
H7v2a – Keras – sous-bassin + bien + territoire (24 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       799.6             732.6                   -9.1                            38.7          B
      maison         2023       836.8             774.2                   -8.1                            38.4          B
 appartement         2023       472.1             166.2                 -184.0                            15.3          C
      maison         2023       450.6             111.7                 -303.3                             9.0          C
 appartement         2024       847.1             744.9                  -13.7                            40.2          B
      maison         2024       747.9             740.6    

année jugée                                                    2023  2024
modèle                                           type de bien            
H7 – Elastic Net – sous-bassin + bien            appartement   -8.9  -5.6
                                                 maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                appartement   -8.7 -12.5
                                                 maison       -12.3  -8.7
H7v1b – Keras – sous-bassin + bien + coordonnées appartement   -9.0 -12.4
                                                 maison       -13.1  -7.8
H7v2a – Keras – sous-bassin + bien + territoire  appartement   -9.7 -11.9
                                                 maison       -13.5  -7.7

**Résultats :** le territoire n'ajoute rien à l'identité du sous-bassin en 2023 : l'erreur de H7v2a est celle de H7v1 à 1 €/m² près (évaluation B). En 2024, elle baisse (−24 €/m² pour les appartements, −12 €/m² pour les maisons en évaluation B), et ce gain est net d'après les tests (section 5.2), mais il ne se retrouve pas en 2023. Le biais reste le même (de −8 à −14 %). Les variables de territoire sont des photos fixes, et l'embedding apprend déjà ce qui est propre et constant pour chaque sous-bassin.

### 4.6 H7v2b : réseau Keras, bien + territoire, sans embedding
Le sous-bassin n'est plus donné au réseau : les variables de territoire le remplacent. Question : le territoire suffit-il à décrire le lieu ? Si H7v2b fait aussi bien que H7v1, le lieu se résume à ces 11 variables ; s'il fait nettement moins bien, l'identité du sous-bassin porte une information que le territoire ne décrit pas.

In [39]:
H7V2B = lancer("H7v2b – Keras – bien + territoire, sans embedding", "keras",
               BLOC_BIEN + BLOC_TERRITOIRE, colonne_embedding=None)
biais([REFERENCE_OU, H7V1, H7V1B, H7V2A, H7V2B])

H7v2b – Keras – bien + territoire, sans embedding | fenêtre 2023 : 5.7 min
H7v2b – Keras – bien + territoire, sans embedding | fenêtre 2024 : 8.7 min
H7v2b – Keras – bien + territoire, sans embedding (23 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       821.3             732.6                  -12.1                            40.2          B
      maison         2023       847.8             774.2                   -9.5                            38.8          B
 appartement         2023       490.2             166.2                 -194.9                            15.8          C
      maison         2023       464.2             111.7                 -315.6                             9.6          C
 appartement         2024       850.8             744.9                  -14.2                            38.9          B
      maison         2024       768.4             740

année jugée                                                     2023  2024
modèle                                            type de bien            
H7 – Elastic Net – sous-bassin + bien             appartement   -8.9  -5.6
                                                  maison       -11.4  -6.8
H7v1 – Keras – sous-bassin + bien                 appartement   -8.7 -12.5
                                                  maison       -12.3  -8.7
H7v1b – Keras – sous-bassin + bien + coordonnées  appartement   -9.0 -12.4
                                                  maison       -13.1  -7.8
H7v2a – Keras – sous-bassin + bien + territoire   appartement   -9.7 -11.9
                                                  maison       -13.5  -7.7
H7v2b – Keras – bien + territoire, sans embedding appartement   -8.3 -10.6
                                                  maison       -12.4  -8.8

**Résultats :** sans l'identité du sous-bassin, les 11 variables de territoire décrivent presque aussi bien le lieu. En évaluation B, l'erreur de H7v2b s'écarte de celle de H7v1 de −20 à +23 €/m² selon l'année et le type de bien (1 à 3 %) ; H7v2b fait un peu moins bien dans trois cas sur quatre. Le biais reste le même (de −8 à −12 %). L'embedding garde donc un léger avantage, mais l'essentiel de ce qui distingue les sous-bassins pour le prix d'une vente est décrit par le parc de logements, la vacance et la densité. La version suivante part de H7v2a, la meilleure des deux.

### 4.7 Le « quand » : prix N-1
On ajoute le prix médian N-1 du sous-bassin, la seule information des modèles sur le niveau récent du marché local. Il entre en log (`log_prix_n1`), dans la table et dans les ventes, pour le réseau comme pour l'Elastic Net. C'est la structure du modèle pont : le log du prix expliqué par le log du prix N-1.

In [8]:
# Prix N-1 en log, dans la table (panier fixe) et dans les ventes
table["log_prix_n1"] = np.log(table["prix_n1"])
ventes["log_prix_n1"] = np.log(ventes["prix_n1"])
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)

#### Export des ventes préparées (pour l'optimisation du réseau)
Les ventes, avec toutes les variables préparées (bien, territoire, prix N-1 en log), sont enregistrées dans `ventes_h7.parquet`, avec la liste des variables de H7v3 dans `variables_h7v3.json`. Le notebook `H7_optimisation.ipynb` les lit : il règle le réseau sur exactement les mêmes variables que H7v3, sans recopier la préparation.

In [9]:
import json

VARIABLES_H7V3 = ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"]

ventes.to_parquet(commun.DATA_OUT / "ventes_h7.parquet", index=False)
with open(commun.DATA_OUT / "variables_h7v3.json", "w", encoding="utf-8") as f:
    json.dump(VARIABLES_H7V3, f, ensure_ascii=False, indent=1)

taille = (commun.DATA_OUT / "ventes_h7.parquet").stat().st_size / 1e6
print(f"Exporté : ventes_h7.parquet ({len(ventes)} ventes, {ventes.shape[1]} colonnes, {taille:.0f} Mo)")
print(f"Variables de H7v3 : {len(VARIABLES_H7V3)}")

Exporté : ventes_h7.parquet (693168 ventes, 69 colonnes, 62 Mo)
Variables de H7v3 : 25


### 4.8 Référence du « quand » : Elastic Net, sous-bassin + bien + territoire + prix N-1
Mêmes colonnes que H7v3. Avec le log du prix N-1 comme variable, apprendre le log du prix revient à apprendre l'évolution par rapport au prix N-1 : le coefficient du log du prix N-1 s'en charge. Un coefficient proche de 1 veut dire que le prix suit celui de l'année précédente, comme pour la naïve (même lecture que pour le modèle pont, H4).

In [41]:
REFERENCE_QUAND = lancer("H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1", "elastic",
                         ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"])

# Réglage retenu et coefficient du log du prix N-1, remis à l'échelle d'origine (comme coef_prix_n1_ du gabarit pont)
for annee in commun.ANNEES_JUGEES:
    m = ajustes[(REFERENCE_QUAND, annee)]
    echelle_n1 = m.echelle_.scale_[list(m.coefs_.index).index("log_prix_n1")]
    print(f"Fenêtre {annee} | réglage : alpha {m.alpha:.0e}, l1_ratio {m.l1_ratio} "
          f"| coefficient du log du prix N-1 : {m.coefs_['log_prix_n1'] / echelle_n1:.3f}")

H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 | fenêtre 2023 : 0.3 min
H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 | fenêtre 2024 : 0.3 min
H7 – Elastic Net – sous-bassin + bien + territoire + prix N-1 (25 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       718.0             732.6                    2.0                            60.0          B
      maison         2023       756.0             774.2                    2.3                            61.5          B
 appartement         2023       217.8             166.2                  -31.1                            32.9          C
      maison         2023       155.2             111.7                  -38.9                            31.5          C
 appartement         2024       731.1             744.9                    1.8                            63.5          B
      maison     

**Résultats :**
- Avec le prix N-1, l'Elastic Net fait un peu mieux que la naïve pour chaque vente (évaluation B) : gain de 1,8 à 2,6 %, et il fait mieux dans 60 à 64 % des sous-bassins. Le gain reste sous le seuil de 3 % de la règle de décision.
- Pour la médiane d'un sous-bassin (évaluation C), il perd en 2023 (−31 % pour les appartements, −39 % pour les maisons) et pour les appartements en 2024 (−22 %), mais il bat la naïve pour les maisons en 2024 : 121 contre 129 €/m² (+6,5 %).
- Le coefficient du log du prix N-1 vaut 0,84 (fenêtre 2023) puis 0,79 (fenêtre 2024). Sous 1, il rapproche de la moyenne les prix des territoires chers et des territoires bon marché, comme le modèle pont (H4).
- Le réglage retient un `alpha` plus fort qu'en 4.1 (0,003 puis 0,01) : la pénalité sert davantage quand plusieurs variables (prix N-1, sous-bassin, territoire) disent le niveau des prix du lieu.

### 4.9 H7v3 : réseau Keras, H7v2a + prix N-1
H7v2a (sous-bassin, bien, territoire) avec le prix N-1. Question : le niveau récent du marché local corrige-t-il l'erreur de niveau des versions du « où » ?

In [42]:
H7V3 = lancer("H7v3 – Keras – sous-bassin + bien + territoire + prix N-1", "keras",
              ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"])
biais([REFERENCE_OU, H7V1, H7V2A, H7V2B, REFERENCE_QUAND, H7V3])

H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 | fenêtre 2023 : 2.3 min
H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 | fenêtre 2024 : 2.5 min
H7v3 – Keras – sous-bassin + bien + territoire + prix N-1 (25 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       709.7             732.6                    3.1                            62.9          B
      maison         2023       731.9             774.2                    5.5                            68.4          B
 appartement         2023       254.3             166.2                  -53.0                            36.8          C
      maison         2023       194.9             111.7                  -74.5                            29.6          C
 appartement         2024       754.9             744.9                   -1.3                            60.8          B
      maison         2024    

année jugée                                                      2023  2024
modèle                                             type de bien            
H7 – Elastic Net – sous-bassin + bien              appartement   -8.9  -5.6
                                                   maison       -11.4  -6.8
H7 – Elastic Net – sous-bassin + bien + territo... appartement    4.3   5.1
                                                   maison         3.5   5.9
H7v1 – Keras – sous-bassin + bien                  appartement   -8.7 -12.5
                                                   maison       -12.3  -8.7
H7v2a – Keras – sous-bassin + bien + territoire    appartement   -9.7 -11.9
                                                   maison       -13.5  -7.7
H7v2b – Keras – bien + territoire, sans embedding  appartement   -8.3 -10.6
                                                   maison       -12.4  -8.8
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    2.6  -2.8
                                                   maison         1.9   5.6

**Résultats :**
- Avec le prix N-1, le réseau bat la naïve pour chaque vente (évaluation B) : gain de 3,1 % (appartements) et 5,5 % (maisons) en 2023, de 6,2 % pour les maisons en 2024 ; il perd un peu pour les appartements en 2024 (−1,3 %). Il fait mieux que la naïve dans 61 à 70 % des sous-bassins.
- En évaluation C, il perd encore contre la naïve (de −34 à −104 %). Dans la médiane d'un territoire, l'effet du bien disparaît : il ne reste qu'à prévoir l'évolution du marché.
- À informations égales, le réseau fait mieux que l'Elastic Net pour chaque vente (évaluation B), sauf pour les appartements en 2024 ; l'Elastic Net fait nettement mieux pour la médiane d'un sous-bassin (évaluation C : 155 contre 195 €/m² pour les maisons en 2023, 121 contre 173 en 2024). Hypothèse : le réseau suit mieux le bien, mais ses prévisions varient plus d'une vente à l'autre, ce qui pèse dans la médiane.
- Le biais : l'Elastic Net surestime les prix de 3,5 à 5,9 % ; il prolonge la hausse apprise sur 2017-2022 (même constat que pour le modèle pont, H4). Le réseau est moins biaisé en moyenne (de −2,8 à +5,6 %).
- Comme attendu, le modèle par vente n'anticipe pas mieux le retournement : il sépare l'effet du bien de celui du marché, mais ne prévoit pas mieux le marché.

### 4.10 Le « quand » du niveau 2 : préparation des indicateurs
Les hypothèses H5 (volume et élan) et H6 (capacité d'achat) sont reprises sur les ventes, avec le réseau. Chaque version part de H7v3 (sous-bassin, bien, territoire, prix N-1) et ajoute un bloc d'indicateurs connus avant l'année jugée.
- **Volume et élan** (préparation P9), avec les indicateurs retenus par l'analyse préalable de H5 : nombre de ventes N-1, rotation du parc du même type (ventes N-1 rapportées au parc du même type) et élan (évolution du prix médian de N-2 à N-1). L'évolution du nombre de ventes, qui ne sépare pas les territoires en hausse, stables et en baisse, et la rotation du parc total, redondante, ne sont pas reprises. Quand le territoire avait moins de 20 ventes en N-2, l'élan est inconnu : il est mis à 0, et une indicatrice le signale. Aucune vente n'est retirée : toutes les versions sont jugées sur les mêmes ventes.
- **Capacité d'achat** (préparation P7) : taux d'effort, mensualité d'un achat type au prix N-1, au taux de décembre N-1, rapportée au revenu local.
- Le nombre de ventes, la rotation et le taux d'effort, très asymétriques, passent en log.

Les indicateurs sont joints aux ventes et à la table (pour le panier fixe), sur l'année, le sous-bassin et le type de bien.

In [45]:
CLES_INDICATEURS = ["annee", "sous_bassin_de_vie", "type_bien"]


def cles_de(t):
    """Année, sous-bassin et type de bien de chaque ligne, dans l'ordre des lignes (ventes : colonnes ; table : index)."""
    if "sous_bassin_de_vie" in t.columns:
        return t[CLES_INDICATEURS].reset_index(drop=True)
    return t.index.to_frame(index=False)[CLES_INDICATEURS]


def ajouter(t, valeurs, colonnes):
    """Ajoute à t (ventes ou table) les colonnes de `valeurs`, jointes sur l'année, le sous-bassin et le type de bien."""
    t = t.copy()
    k = cles_de(t).merge(valeurs[CLES_INDICATEURS + colonnes], on=CLES_INDICATEURS, how="left", validate="many_to_one")
    t[colonnes] = k[colonnes].to_numpy()
    return t


# Volume et élan (P9) : indicateurs retenus par l'analyse préalable de H5. L'élan est vide quand le territoire avait
# moins de 20 ventes en N-2 (appartements surtout) : vide -> 0 et indicatrice evol_n2_inconnue = 1, aucune vente retirée
P9 = pd.read_parquet(commun.DATA_OUT / "p9_volume_elan.parquet")
COLONNES_P9 = ["evol_prix_n2_n1", "rotation_parc_type"]
# Capacité d'achat (P7)
P7 = pd.read_parquet(commun.DATA_OUT / "fe_capacite_achat_sbv.parquet")


def preparer_quand_niveau2(t):
    """Indicateurs P9 et P7 ajoutés à t ; asymétriques en log ; élan vide remplacé par 0 avec une indicatrice."""
    t = ajouter(ajouter(t, P9, COLONNES_P9), P7, ["taux_effort"])
    t["evol_n2_inconnue"] = t["evol_prix_n2_n1"].isna().astype(int)
    t["evol_prix_n2_n1"] = t["evol_prix_n2_n1"].fillna(0)
    for v in ["nb_ventes_n1", "rotation_parc_type", "taux_effort"]:
        assert (t[v] > 0).all(), f"{v} : valeur vide, nulle ou négative"
        t["log_" + v] = np.log(t[v])
    return t


table = preparer_quand_niveau2(table)          # pour le panier fixe (évaluation C)
ventes = preparer_quand_niveau2(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)

BLOC_H5 = ["log_nb_ventes_n1", "log_rotation_parc_type", "evol_prix_n2_n1", "evol_n2_inconnue"]
BLOC_H6 = ["log_taux_effort"]
BASE_V3 = ["sous_bassin_de_vie"] + BLOC_BIEN + BLOC_TERRITOIRE + ["log_prix_n1"]
assert ventes[BLOC_H5 + BLOC_H6].notna().all().all() and table[BLOC_H5 + BLOC_H6].notna().all().all()
print("Ventes 2017-2024 dont l'élan est inconnu (%) :")
print(ventes[ventes["annee"] >= ANNEE_DEBUT_H7].groupby("type_bien")["evol_n2_inconnue"].mean().mul(100).round(2).to_string())

Ventes 2017-2024 dont l'élan est inconnu (%) :
type_bien
appartement    0.27
maison         0.01


**Constat :** l'élan est inconnu pour 0,27 % des ventes d'appartements et 0,01 % des ventes de maisons (2017-2024) : l'indicatrice concerne très peu de ventes.

### 4.11 H7vH5 : réseau Keras, H7v3 + volume et élan
Hypothèse H5 sur les ventes : le volume de ventes et l'élan des prix de l'année précédente annoncent-ils le sens de l'évolution ?

In [46]:
H7VH5 = lancer("H7vH5 – Keras – H7v3 + volume et élan", "keras", BASE_V3 + BLOC_H5)

H7vH5 – Keras – H7v3 + volume et élan | fenêtre 2023 : 3.0 min
H7vH5 – Keras – H7v3 + volume et élan | fenêtre 2024 : 2.6 min
H7vH5 – Keras – H7v3 + volume et élan (29 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       715.9             732.6                    2.3                            59.7          B
      maison         2023       738.9             774.2                    4.6                            65.7          B
 appartement         2023       270.6             166.2                  -62.8                            34.1          C
      maison         2023       204.0             111.7                  -82.6                            28.4          C
 appartement         2024       753.5             744.9                   -1.2                            61.6          B
      maison         2024       696.7             740.6                    5.9           

**Résultats :** le volume et l'élan ne changent presque rien à H7v3. En évaluation B, l'erreur bouge de −1 à +7 €/m² (moins de 1 %) ; en évaluation C, elle augmente un peu en 2023 (+16 €/m² pour les appartements, +9 €/m² pour les maisons). Le biais des maisons en 2024 baisse (de +5,6 % à +3,7 %), sans que l'erreur moyenne baisse. Contre la naïve, H7vH5 est retenu pour les maisons en évaluation B, comme H7v3 (section 5.1) ; contre H7v3, aucune année n'est réussie (section 5.2).

### 4.12 H7vH6 : réseau Keras, H7v3 + capacité d'achat
Hypothèse H6 sur les ventes : le taux d'effort d'un achat type, qui suit la remontée des taux des crédits, annonce-t-il le retournement ?

In [47]:
H7VH6 = lancer("H7vH6 – Keras – H7v3 + capacité d'achat", "keras", BASE_V3 + BLOC_H6)

H7vH6 – Keras – H7v3 + capacité d'achat | fenêtre 2023 : 3.2 min
H7vH6 – Keras – H7v3 + capacité d'achat | fenêtre 2024 : 2.3 min
H7vH6 – Keras – H7v3 + capacité d'achat (26 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       721.1             732.6                    1.6                            58.3          B
      maison         2023       751.7             774.2                    2.9                            57.9          B
 appartement         2023       294.5             166.2                  -77.2                            29.6          C
      maison         2023       247.9             111.7                 -121.9                            21.1          C
 appartement         2024       764.3             744.9                   -2.6                            58.0          B
      maison         2024       705.5             740.6                    4.7     

**Résultats :** la capacité d'achat déplace le niveau des prévisions sans les améliorer. Le biais des maisons en 2024 disparaît (de +5,6 % pour H7v3 à 0,0 %) : le taux d'effort contient la remontée du taux des crédits. Mais l'erreur augmente partout : de 9 à 20 €/m² en évaluation B (1 à 3 %), de 25 à 53 €/m² en évaluation C. Le réseau sous-estime maintenant les maisons en 2023 (−2,6 %) et les appartements en 2024 (−5,5 %). Corriger le décalage moyen d'une année ne suffit pas : les écarts entre territoires et entre biens sont moins bien prévus. Contre la naïve, H7vH6 ne réussit qu'une année pour les maisons en évaluation B (2023 : gain de 2,9 %, sous le seuil de 3 %).

### 4.13 Les trimestres précédents : préparation
Les indicateurs annuels ne voient le retournement qu'avec un an de retard. Les dates des ventes permettent de regarder la fin de l'année N-1 : un ralentissement du prix ou une chute du volume au second semestre peut annoncer le retournement de l'année N.
- Un sous-bassin compte trop peu de ventes de l'échantillon par trimestre. Le calcul se fait au **département** (deux premiers caractères du code du sous-bassin) et par type de bien.
- Pour l'année N : prix médian (pondéré par `w_h`) et volume estimé (somme des `w_h`) des **trimestres 3 et 4 de N-1**, comparés aux mêmes trimestres de N-2 (évolution sur un an, en %). Comparer un trimestre au même trimestre de l'année d'avant retire la saisonnalité.
- Si un département compte moins de 30 ventes dans l'un des trimestres comparés, la valeur nationale du type de bien la remplace ; une indicatrice le signale.
- Seules des ventes de N-1 et N-2 servent : rien de l'année jugée.

In [49]:
from commun import mediane_ponderee

SEUIL_TRIMESTRE = 30          # ventes de l'échantillon par département, type de bien et trimestre
TRIMESTRES = [3, 4]           # les deux derniers trimestres de N-1


def stats_trimestre(v, lieu):
    """Prix médian pondéré (w_h), volume estimé (somme des w_h) et nombre de ventes, par lieu, année et trimestre."""
    g = v.groupby(lieu + ["anneemut", "trimestre"])
    return pd.DataFrame({"mediane": g.apply(lambda e: mediane_ponderee(e["prix_m2"], e["w_h"]), include_groups=False),
                         "volume": g["w_h"].sum(), "ventes": g.size()}).reset_index()


def glissements(s, lieu):
    """Pour l'année N : évolution du trimestre q de N-1 par rapport au trimestre q de N-2 (prix médian et volume, en %)."""
    sorties = []
    for q in TRIMESTRES:
        e = s[s["trimestre"] == q].drop(columns="trimestre")
        n1 = e.assign(annee=e["anneemut"] + 1).drop(columns="anneemut")     # trimestre q de N-1, rattaché à l'année N
        n2 = e.assign(annee=e["anneemut"] + 2).drop(columns="anneemut")     # trimestre q de N-2, rattaché à l'année N
        m = n1.merge(n2, on=lieu + ["annee"], suffixes=("_n1", "_n2"))
        sorties.append(pd.DataFrame({f"gliss_prix_t{q}": (m["mediane_n1"] / m["mediane_n2"] - 1).to_numpy() * 100,
                                     f"gliss_volume_t{q}": (m["volume_n1"] / m["volume_n2"] - 1).to_numpy() * 100,
                                     f"fiable_t{q}": (m[["ventes_n1", "ventes_n2"]].min(axis=1) >= SEUIL_TRIMESTRE).to_numpy()},
                                    index=pd.MultiIndex.from_frame(m[lieu + ["annee"]])))
    return pd.concat(sorties, axis=1).reset_index()


assert ventes_brutes["anneemut"].max() <= commun.ANNEES_JUGEES[-1], "aucune vente de 2025 ne doit servir"
vb = ventes_brutes.assign(departement=ventes_brutes["sous_bassin_de_vie"].str[:2],
                          trimestre=(ventes_brutes["moismut"] - 1) // 3 + 1)
G_DEP = glissements(stats_trimestre(vb, ["departement", "type_bien"]), ["departement", "type_bien"])
G_NAT = glissements(stats_trimestre(vb, ["type_bien"]), ["type_bien"])
BLOC_TRIMESTRES = [f"gliss_{x}_t{q}" for q in TRIMESTRES for x in ("prix", "volume")]


def preparer_trimestres(t):
    """Glissements des trimestres 3 et 4 de N-1 du département ; valeur nationale si moins de 30 ventes."""
    t = t.copy()
    k = cles_de(t).assign(departement=lambda d: d["sous_bassin_de_vie"].str[:2])
    k = k.merge(G_DEP, on=["departement", "type_bien", "annee"], how="left", validate="many_to_one")
    k = k.merge(G_NAT, on=["type_bien", "annee"], how="left", validate="many_to_one", suffixes=("", "_nat"))
    for q in TRIMESTRES:
        national = ~k[f"fiable_t{q}"].eq(True)        # département absent ou trop petit : valeur nationale
        for x in ("prix", "volume"):
            c = f"gliss_{x}_t{q}"
            k.loc[national, c] = k.loc[national, c + "_nat"]
    t[BLOC_TRIMESTRES] = k[BLOC_TRIMESTRES].to_numpy(dtype=float)
    t["trimestre_national"] = (~k[[f"fiable_t{q}" for q in TRIMESTRES]].eq(True).all(axis=1)).astype(int).to_numpy()
    return t


table = preparer_trimestres(table)
ventes = preparer_trimestres(ventes)
FENETRES = fenetres(ventes, annee_debut=ANNEE_DEBUT_H7)
assert ventes[BLOC_TRIMESTRES].notna().all().all() and table[BLOC_TRIMESTRES].notna().all().all()

print("Ventes avec au moins une valeur nationale (département trop petit), en % :")
print(ventes.groupby("annee")["trimestre_national"].mean().mul(100).round(1).loc[ANNEE_DEBUT_H7:].to_string())
print("\nGlissements nationaux (%), par type de bien et année de prix :")
print(G_NAT[G_NAT["annee"].between(ANNEE_DEBUT_H7, commun.ANNEES_JUGEES[-1])]
      .set_index(["type_bien", "annee"])[BLOC_TRIMESTRES].round(1).to_string())

Ventes avec au moins une valeur nationale (département trop petit), en % :
annee
2017    3.9
2018    4.4
2019    3.8
2020    3.6
2021    3.2
2022    3.2
2023    3.1
2024    4.3

Glissements nationaux (%), par type de bien et année de prix :
                   gliss_prix_t3  gliss_volume_t3  gliss_prix_t4  gliss_volume_t4
type_bien   annee                                                                
appartement 2017            -2.2              7.7            0.1              0.1
            2018             3.2              5.6            2.4              8.1
            2019             2.4              3.7           -0.7              6.7
            2020             0.6             10.8            3.1             10.8
            2021             2.8            -13.9            3.4             11.7
            2022             7.1             31.2            7.4             -6.9
            2023             6.8             -3.5            6.2             -3.8
            2024     

**Constat :**
- De 3,1 à 4,4 % des ventes, selon l'année, reçoivent au moins une valeur nationale (département trop petit pour un trimestre).
- Pour les prix 2023 (fin 2022), les prix des trimestres 3 et 4 montaient encore de 6 à 9 % sur un an ; seul le volume des maisons reculait (−11 %), celui des appartements à peine (−4 %). Le signal précurseur du retournement de 2023 est donc faible.
- Pour les prix 2024 (fin 2023), le signal est net : volume en baisse de 20 à 27 %, prix du 4e trimestre déjà en baisse (−3 % pour les appartements, −6 % pour les maisons).
- Dans les années d'apprentissage, le volume a déjà reculé sans baisse des prix ensuite : au 3e trimestre 2020 (prix 2021), après le confinement, de −10 à −14 %.

### 4.14 H7vT : réseau Keras, H7v3 + trimestres précédents
H7v3 avec, en plus, l'évolution du prix et du volume des trimestres 3 et 4 de l'année précédente. Question : la fin de l'année précédente annonce-t-elle le retournement mieux que les indicateurs annuels ?

In [50]:
H7VT = lancer("H7vT – Keras – H7v3 + trimestres précédents", "keras",
              BASE_V3 + BLOC_TRIMESTRES + ["trimestre_national"])
biais([H7V3, H7VH5, H7VH6, H7VT])

H7vT – Keras – H7v3 + trimestres précédents | fenêtre 2023 : 2.2 min
H7vT – Keras – H7v3 + trimestres précédents | fenêtre 2024 : 2.3 min
H7vT – Keras – H7v3 + trimestres précédents (30 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       712.5             732.6                    2.7                            60.7          B
      maison         2023       735.7             774.2                    5.0                            67.7          B
 appartement         2023       255.8             166.2                  -53.9                            35.2          C
      maison         2023       205.9             111.7                  -84.3                            28.8          C
 appartement         2024       755.8             744.9                   -1.5                            60.7          B
      maison         2024       696.8             740.6                

année jugée                                                      2023  2024
modèle                                             type de bien            
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    2.6  -2.8
                                                   maison         1.9   5.6
H7vH5 – Keras – H7v3 + volume et élan              appartement    2.4  -4.1
                                                   maison         1.8   3.7
H7vH6 – Keras – H7v3 + capacité d'achat            appartement    0.6  -5.5
                                                   maison        -2.6  -0.0
H7vT – Keras – H7v3 + trimestres précédents        appartement    1.7  -4.9
                                                   maison         0.6   2.0

**Résultats :** les trimestres précédents vont dans le bon sens pour les maisons sans réduire l'erreur. Le biais des maisons baisse de plus de moitié (2024 : de +5,6 % à +2,0 % ; 2023 : de +1,9 % à +0,6 %), mais l'erreur reste celle de H7v3 à 1 à 4 €/m² près en évaluation B, et augmente un peu en évaluation C (+2 à +11 €/m²). Pour les appartements, le biais de 2024 augmente (−4,9 % contre −2,8 %). Hypothèses : le réseau n'a jamais vu, à l'apprentissage, une chute du volume de 25 % accompagnée d'une baisse des prix ; et il y a vu une baisse du volume (fin 2020) qui n'annonçait pas de baisse des prix. Contre la naïve, H7vT est retenu pour les maisons en évaluation B, comme H7v3.

### 4.15 H7v4 : réseau Keras, le « quand » complet
H7v3 avec les trois blocs du « quand » réunis : volume et élan, capacité d'achat, trimestres précédents. Question : les indicateurs se complètent-ils ? Sur la table agrégée, l'élan aide surtout en 2023 (année où la hausse s'arrête) et le taux d'effort surtout en 2024 (année où le crédit renchérit).

In [51]:
H7V4 = lancer("H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres", "keras",
              BASE_V3 + BLOC_H5 + BLOC_H6 + BLOC_TRIMESTRES + ["trimestre_national"])
biais([H7V3, H7VH5, H7VH6, H7VT, H7V4])

H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres | fenêtre 2023 : 3.0 min
H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres | fenêtre 2024 : 3.1 min
H7v4 – Keras – H7v3 + volume, élan, capacité d'achat et trimestres (35 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%) évaluation
 appartement         2023       713.0             732.6                    2.7                            59.1          B
      maison         2023       739.4             774.2                    4.5                            65.4          B
 appartement         2023       248.6             166.2                  -49.6                            37.5          C
      maison         2023       202.5             111.7                  -81.2                            31.0          C
 appartement         2024       763.9             744.9                   -2.5                            60.2          B
  

année jugée                                                      2023  2024
modèle                                             type de bien            
H7v3 – Keras – sous-bassin + bien + territoire ... appartement    2.6  -2.8
                                                   maison         1.9   5.6
H7v4 – Keras – H7v3 + volume, élan, capacité d'... appartement    3.7  -5.6
                                                   maison         1.7   1.2
H7vH5 – Keras – H7v3 + volume et élan              appartement    2.4  -4.1
                                                   maison         1.8   3.7
H7vH6 – Keras – H7v3 + capacité d'achat            appartement    0.6  -5.5
                                                   maison        -2.6  -0.0
H7vT – Keras – H7v3 + trimestres précédents        appartement    1.7  -4.9
                                                   maison         0.6   2.0

**Résultats :** réunis, les indicateurs du « quand » ne font pas mieux que séparément. L'erreur augmente un peu par rapport à H7v3 : de 3 à 9 €/m² en évaluation B ; en évaluation C, elle baisse pour les appartements en 2023 (−6 €/m²) et augmente ailleurs (+8 à +20 €/m²). Le biais des maisons en 2024 baisse (de +5,6 % à +1,2 %), celui des appartements augmente (−5,6 % contre −2,8 %). Contre la naïve, H7v4 est retenu pour les maisons en évaluation B, comme H7v3.

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs**, déjà affichées après chaque modèle : MAE du modèle et de la naïve en €/m², gain sur la naïve, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux). En évaluation B, l'erreur d'un sous-bassin est la moyenne des erreurs de ses ventes, pondérée par `w_h`.
   - **IC 95 %** (bootstrap par bassin de vie) : si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins ? On la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout).
3. **La règle de décision** : une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit les deux années jugées.

Le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau : deux entraînements avec des graines différentes donneraient des résultats un peu différents.

In [52]:
pd.set_option("display.max_rows", 200)      # tableaux affichés en entier
res = pd.DataFrame(resultats)
tst = pd.DataFrame(tests)
tst[["modèle", "type de bien", "année jugée", "écart de MAE avec la naïve (€/m²)",
     "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",appartement,2023,112.203,94.562,129.776,0.0,0.521
1,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",maison,2023,86.792,70.337,105.833,0.0,0.545
2,"H7 – Elastic Net – sous-bassin + bien (C, pani...",appartement,2023,331.695,307.398,357.684,0.0,0.854
3,"H7 – Elastic Net – sous-bassin + bien (C, pani...",maison,2023,347.170,317.855,382.873,0.0,0.950
4,"H7 – Elastic Net – sous-bassin + bien (B, ventes)",appartement,2024,96.270,76.630,115.800,0.0,0.416
...,...,...,...,...,...,...,...,...
83,"H7v4 – Keras – H7v3 + volume, élan, capacité d...",maison,2023,90.745,80.441,101.885,0.0,0.549
84,"H7v4 – Keras – H7v3 + volume, élan, capacité d...",appartement,2024,18.984,-23.103,90.823,0.0,-0.171
85,"H7v4 – Keras – H7v3 + volume, élan, capacité d...",maison,2024,-37.468,-43.203,-30.985,0.0,-0.393
86,"H7v4 – Keras – H7v3 + volume, élan, capacité d...",appartement,2024,172.385,95.288,299.111,0.0,0.511


**Comment lire le tableau des tests :** prenons l'Elastic Net du « où », appartements, 2023, évaluation B. L'écart de MAE avec la naïve vaut +112 €/m² : le modèle se trompe en moyenne de 112 €/m² de plus que la naïve par sous-bassin. L'IC 95 % va de +95 à +130 €/m², entièrement au-dessus de 0 : le modèle fait vraiment moins bien. La rang-bisériale vaut +0,52 : il fait moins bien dans une nette majorité de sous-bassins. À l'inverse, H7v4, maisons, 2024, évaluation B : l'écart vaut −37 €/m², l'IC va de −43 à −31 €/m², entièrement sous 0, et la rang-bisériale vaut −0,39 : le modèle fait vraiment mieux que la naïve, dans une majorité de sous-bassins.

### 5.1 Règle de décision contre la naïve

In [53]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la comparaison, fixé avant les résultats


def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"})


regle_decision(res, tst)

années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7 – Elastic Net – sous-bassin + bien (B, ventes)  appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien (C, panie... appartement                 0              2   False
                                                   maison                      0              2   False
H7v1 – Keras – sous-bassin + bien (B, ventes)      appartement                 0              2   False
                                                   maison                      0              2   False
H7v1 – Keras – sous-bassin + bien (C, panier fixe) appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b – Keras – sous-bassin + bien + coordonnée... appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b – Keras – sous-bassin + bien + coordonnée... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a – Keras – sous-bassin + bien + territoire... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a – Keras – sous-bassin + bien + territoire... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b – Keras – bien + territoire, sans embeddi... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b – Keras – bien + territoire, sans embeddi... appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      0              2   False
H7 – Elastic Net – sous-bassin + bien + territo... appartement                 0              2   False
                                                   maison                      1              2   False
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 1              2   False
                                                   maison                      2              2    True
H7v3 – Keras – sous-bassin + bien + territoire ... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH5 – Keras – H7v3 + volume et élan (B, ventes)  appartement                 0              2   False
                                                   maison                      2              2    True
H7vH5 – Keras – H7v3 + volume et élan (C, panie... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH6 – Keras – H7v3 + capacité d'achat (B, ven... appartement                 0              2   False
                                                   maison                      1              2   False
H7vH6 – Keras – H7v3 + capacité d'achat (C, pan... appartement                 0              2   False
                                                   maison                      0              2   False
H7vT – Keras – H7v3 + trimestres précédents (B,... appartement                 0              2   False
         

**Résultats contre la naïve :**
- **Modèles du « où »** (Elastic Net, H7v1, H7v1b, H7v2a, H7v2b) : aucune année réussie, ni en évaluation B ni en évaluation C, pour les deux types de bien.
- **Elastic Net du « quand »** : en évaluation B, son gain (de 1,8 à 2,6 %) reste sous le seuil de 3 % ; en évaluation C, il réussit une année, les maisons en 2024 (+6,5 %). Non retenu.
- **Réseaux avec le prix N-1** (H7v3, H7vH5, H7vT, H7v4) : retenus pour les maisons en évaluation B, les deux années. Pour les appartements, H7v3 ne réussit que 2023. H7vH6 ne réussit que 2024 pour les maisons (gain de 2,9 % en 2023).
- **En évaluation C**, aucun réseau ne réussit une seule année.
- Le modèle par vente bat donc la naïve pour prévoir le prix d'une maison vendue, à condition de connaître le prix N-1, mais pas pour prévoir le prix médian d'un territoire.

### 5.2 Les versions entre elles
Chaque comparaison répond à une question du plan ; on applique les mêmes mesures, les mêmes tests et la même règle de décision, en remplaçant la naïve par la prévision du modèle de comparaison :
- en évaluation B, la prévision de la référence remplace le prix N-1 de chaque vente ;
- en évaluation C, la médiane des prévisions de la référence remplace le prix N-1 du sous-bassin.

| Comparaison | Question |
|---|---|
| H7v1 contre l'Elastic Net du « où » | À informations égales, le réseau fait-il mieux qu'un Elastic Net ? |
| H7v1b contre H7v1 | La localisation à l'intérieur du sous-bassin apporte-t-elle quelque chose ? |
| H7v2a contre H7v1 | Le territoire ajoute-t-il quelque chose à l'identité du sous-bassin ? |
| H7v2b contre H7v1 | Le territoire peut-il remplacer l'identité du sous-bassin ? |
| H7v3 contre l'Elastic Net du « quand » | Même question que la première, avec le prix N-1 |
| H7v3 contre H7v2a | Qu'apporte le prix N-1 ? |
| H7vH5 contre H7v3 | Le volume et l'élan apportent-ils quelque chose au prix N-1 (H5) ? |
| H7vH6 contre H7v3 | La capacité d'achat apporte-t-elle quelque chose au prix N-1 (H6) ? |
| H7vT contre H7v3 | Les trimestres précédents apportent-ils quelque chose au prix N-1 ? |
| H7v4 contre H7v3 | Les indicateurs du « quand » réunis apportent-ils quelque chose au prix N-1 ? |

Lecture : un gain positif et un IC entièrement sous 0 veulent dire que le modèle fait vraiment mieux que sa comparaison.

In [58]:
from commun import CLES, mediane_ponderee, evaluer, tester


def medianes_panier(panier, prevision):
    """Médiane pondérée (w_h) des prévisions du panier fixe, par sous-bassin (comme evaluer_panier)."""
    d = pd.DataFrame({"p": prevision.loc[panier.index], "w": panier["w_h"], **{c: panier[c] for c in CLES}})
    return d.groupby(table.index.names).apply(lambda g: mediane_ponderee(g["p"], g["w"]), include_groups=False)


COMPARAISONS = [  # (libellé, modèle, modèle de comparaison)
    ("H7v1 contre l'Elastic Net du « où »", H7V1, REFERENCE_OU),
    ("H7v1b contre H7v1 (coordonnées)", H7V1B, H7V1),
    ("H7v2a contre H7v1 (territoire en plus)", H7V2A, H7V1),
    ("H7v2b contre H7v1 (territoire à la place du sous-bassin)", H7V2B, H7V1),
    ("H7v3 contre l'Elastic Net du « quand »", H7V3, REFERENCE_QUAND),
    ("H7v3 contre H7v2a (prix N-1 en plus)", H7V3, H7V2A),
    ("H7vH5 contre H7v3 (volume et élan en plus)", H7VH5, H7V3),
    ("H7vH6 contre H7v3 (capacité d'achat en plus)", H7VH6, H7V3),
    ("H7vT contre H7v3 (trimestres précédents en plus)", H7VT, H7V3),
    ("H7v4 contre H7v3 (tout le « quand » en plus)", H7V4, H7V3),
]

resultats_ref, tests_ref = [], []
for libelle, nom, nom_ref in COMPARAISONS:
    for annee, (app, ev) in FENETRES.items():
        p, p_panier = previsions[(nom, annee)]
        r, r_panier = previsions[(nom_ref, annee)]
        # B : la prévision de la référence remplace la naïve (prix_n1) pour chaque vente
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = r
        evaluer_ventes(libelle, ev_ref, p, resultats_ref)
        tester_ventes(libelle, ev_ref, p, tests_ref)
        # C : la médiane des prévisions de la référence remplace le prix N-1 du sous-bassin
        panier = panier_fixe(ventes_brutes, table, annee)
        med, med_ref = medianes_panier(panier, p_panier), medianes_panier(panier, r_panier)
        evaluation = table.loc[table.index.intersection(med.index)].copy()
        evaluation["prix_n1"] = med_ref.loc[evaluation.index]
        evaluer(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], resultats_ref)
        tester(f"{libelle} (C, panier fixe)", evaluation, med.loc[evaluation.index], tests_ref)

# Mêmes tableaux que contre la naïve, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2023,798.20,844.83,5.52,66.16
1,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2023,836.91,860.97,2.79,60.62
2,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2023,451.61,497.92,9.30,61.96
3,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2023,443.49,458.88,3.35,57.08
4,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2024,870.69,841.15,-3.51,49.20
...,...,...,...,...,...,...,...
75,H7v4 contre H7v3 (tout le « quand » en plus) (...,maison,2023,202.45,194.89,-3.88,49.94
76,H7v4 contre H7v3 (tout le « quand » en plus) (...,appartement,2024,763.86,754.86,-1.19,47.84
77,H7v4 contre H7v3 (tout le « quand » en plus) (...,maison,2024,703.13,694.95,-1.18,45.01
78,H7v4 contre H7v3 (tout le « quand » en plus) (...,appartement,2024,319.48,299.92,-6.52,45.82


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2023,-46.633,-60.383,-32.712,0.000,-0.381
1,"H7v1 contre l'Elastic Net du « où » (B, ventes)",maison,2023,-24.060,-35.756,-14.915,0.000,-0.256
2,"H7v1 contre l'Elastic Net du « où » (C, panier...",appartement,2023,-46.308,-77.319,-5.446,0.000,-0.291
3,"H7v1 contre l'Elastic Net du « où » (C, panier...",maison,2023,-15.383,-30.416,-2.695,0.000,-0.116
4,"H7v1 contre l'Elastic Net du « où » (B, ventes)",appartement,2024,29.544,0.798,71.172,0.067,0.074
...,...,...,...,...,...,...,...,...
75,H7v4 contre H7v3 (tout le « quand » en plus) (...,maison,2023,7.564,1.295,13.959,0.151,0.039
76,H7v4 contre H7v3 (tout le « quand » en plus) (...,appartement,2024,9.001,-3.801,27.082,0.080,0.071
77,H7v4 contre H7v3 (tout le « quand » en plus) (...,maison,2024,8.185,4.824,11.746,0.000,0.132
78,H7v4 contre H7v3 (tout le « quand » en plus) (...,appartement,2024,19.557,-1.639,49.287,0.026,0.090


#### Règle de décision contre la comparaison

In [59]:
regle_decision(res_ref, tst_ref, comparaison="référence")

années réussies  années jugées  retenu
modèle                                             type de bien                                        
H7v1 contre l'Elastic Net du « où » (B, ventes)    appartement                 1              2   False
                                                   maison                      1              2   False
H7v1 contre l'Elastic Net du « où » (C, panier ... appartement                 1              2   False
                                                   maison                      1              2   False
H7v1b contre H7v1 (coordonnées) (B, ventes)        appartement                 0              2   False
                                                   maison                      0              2   False
H7v1b contre H7v1 (coordonnées) (C, panier fixe)   appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a contre H7v1 (territoire en plus) (B, ventes) appartement                 0              2   False
                                                   maison                      0              2   False
H7v2a contre H7v1 (territoire en plus) (C, pani... appartement                 1              2   False
                                                   maison                      1              2   False
H7v2b contre H7v1 (territoire à la place du sou... appartement                 0              2   False
                                                   maison                      0              2   False
H7v2b contre H7v1 (territoire à la place du sou... appartement                 1              2   False
                                                   maison                      0              2   False
H7v3 contre l'Elastic Net du « quand » (B, ventes) appartement                 0              2   False
                                                   maison                      2              2    True
H7v3 contre l'Elastic Net du « quand » (C, pani... appartement                 0              2   False
                                                   maison                      0              2   False
H7v3 contre H7v2a (prix N-1 en plus) (B, ventes)   appartement                 2              2    True
                                                   maison                      2              2    True
H7v3 contre H7v2a (prix N-1 en plus) (C, panier... appartement                 2              2    True
                                                   maison                      2              2    True
H7vH5 contre H7v3 (volume et élan en plus) (B, ... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH5 contre H7v3 (volume et élan en plus) (C, ... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH6 contre H7v3 (capacité d'achat en plus) (B... appartement                 0              2   False
                                                   maison                      0              2   False
H7vH6 contre H7v3 (capacité d'achat en plus) (C... appartement                 0              2   False
                                                   maison                      0              2   False
H7vT contre H7v3 (trimestres précédents en plus... appartement                 0              2   False
                                                   maison                      0              2   False
H7vT contre H7v3 (trimestres précédents en plus... appartement                 0              2   False
                                                   maison                      0              2   False
H7v4 contre H7v3 (tout le « quand » en plus) (B... appartement                 0              2   False
         

**Comment lire :** prenons « H7v3 contre H7v2a », maisons, 2023, évaluation C. L'écart de MAE avec la référence vaut −256 €/m² : avec le prix N-1, l'erreur sur la médiane d'un sous-bassin baisse de 256 €/m² par rapport au même réseau sans prix N-1 (gain de 57 %). L'IC 95 % va de −273 à −239 €/m², entièrement sous 0, et la rang-bisériale vaut −0,86 : H7v3 fait mieux dans presque tous les sous-bassins.

**Résultats :**
- **Le prix N-1 (H7v3 contre H7v2a)** est la seule comparaison retenue dans les deux évaluations et pour les deux types de bien : gain de 7 à 13 % en évaluation B et de 42 à 57 % en évaluation C, IC entièrement sous 0 et rang-bisériale de −0,55 à −0,86 les deux années.
- **Réseau contre Elastic Net, sans prix N-1 (H7v1 contre l'Elastic Net du « où »)** : le réseau réussit une année sur deux dans chaque cas. Il fait mieux en 2023 pour les appartements (évaluation B : gain de 5,5 %, IC de −60 à −33 €/m² ; évaluation C : gain de 9,3 %) et pour les maisons en 2024 (évaluation B : gain de 3,7 %) ; l'Elastic Net fait mieux pour les appartements en 2024 (écart de +30 €/m² en B, +92 €/m² en C). Avantage léger au réseau, pas régulier : non retenu.
- **Réseau contre Elastic Net, avec le prix N-1 (H7v3 contre l'Elastic Net du « quand »)** : le réseau est retenu pour les maisons en évaluation B (gain de 3,2 % puis 3,6 %, IC sous 0, rang-bisériale −0,35 les deux années). Pour les appartements, l'écart n'est pas net en évaluation B. En évaluation C, l'Elastic Net fait nettement mieux partout (écart de +36 à +121 €/m², IC entièrement au-dessus de 0).
- **Les coordonnées (H7v1b contre H7v1)** : gain minime pour les maisons (−5 à −7 €/m², IC sous 0 dans trois cas sur quatre), soit moins de 1 % ; rien pour les appartements. La localisation à l'intérieur du sous-bassin est presque redondante avec le sous-bassin.
- **Le territoire en plus du sous-bassin (H7v2a contre H7v1)** : aucun effet en 2023 (et une erreur plus grande pour les appartements en évaluation C) ; un gain net en 2024 dans les quatre cas (−12 à −29 €/m², rang-bisériale de −0,25 à −0,35), qui reste sous le seuil de 3 % en évaluation B. Le gain n'est pas régulier d'une année à l'autre : non retenu.
- **Le territoire à la place du sous-bassin (H7v2b contre H7v1)** : erreur plus grande en 2023 (+11 à +39 €/m²) et pour les maisons en 2024, plus petite pour les appartements en 2024. Les 11 variables de territoire décrivent presque aussi bien le lieu que l'identité du sous-bassin, sans la remplacer complètement.
- **Les indicateurs du « quand » du niveau 2 (H7vH5, H7vH6, H7vT et H7v4 contre H7v3)** : aucune année réussie. En évaluation B, le gain sur H7v3 va de −1,0 à +0,2 % (volume et élan), de −0,5 à −0,1 % (trimestres), de −1,2 à −0,5 % (tout le « quand ») et de −2,7 à −1,2 % (capacité d'achat). En évaluation C, l'erreur augmente presque partout, surtout avec la capacité d'achat (de −8 à −27 %). Exemple : H7v4, maisons, 2024, évaluation B : écart de +8 €/m², IC de +5 à +12 €/m², un peu moins bien que H7v3.
- Réserve : le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau. Les petits écarts entre deux réseaux (quelques €/m²) peuvent dépendre de la graine.

## 6. Interprétation

### 6.1 Erreurs de H7v3 selon le niveau des prix du sous-bassin
Évaluation B, par année jugée et type de bien : les sous-bassins sont répartis en quatre groupes de même taille selon leur prix N-1 (quartiles), du moins cher au plus cher. Par groupe : erreur moyenne du modèle et de la naïve, écart moyen (négatif : le modèle fait mieux) et part des sous-bassins où le modèle gagne. Puis, pour les appartements en 2024 (seul cas où H7v3 n'est pas retenu contre la naïve en évaluation B), les dix sous-bassins où le modèle perd le plus.

In [60]:
from commun import CLES, _par_sous_bassin

QUARTILES = ["Q1 (moins chers)", "Q2", "Q3", "Q4 (plus chers)"]
for annee, (app, ev) in FENETRES.items():
    t = _par_sous_bassin(ev, previsions[(H7V3, annee)][0])        # erreurs pondérées par w_h, par sous-bassin
    t["écart (€/m²)"] = t["modele"] - t["naive"]
    t["prix N-1 (€/m²)"] = table["prix_n1"].reindex(pd.MultiIndex.from_frame(t[table.index.names])).to_numpy()
    t["ventes"] = ev.groupby(CLES).size().reindex(pd.MultiIndex.from_frame(t[CLES])).to_numpy()
    for type_bien, e in t.groupby("type_bien"):
        quartile = pd.qcut(e["prix N-1 (€/m²)"], 4, labels=QUARTILES)
        resume = e.groupby(quartile, observed=True).agg(
            **{"sous-bassins": ("écart (€/m²)", "size"),
               "prix N-1 médian (€/m²)": ("prix N-1 (€/m²)", "median"),
               "MAE modèle (€/m²)": ("modele", "mean"), "MAE naïve (€/m²)": ("naive", "mean"),
               "écart moyen (€/m²)": ("écart (€/m²)", "mean"),
               "sous-bassins où le modèle gagne (%)": ("écart (€/m²)", lambda x: (x < 0).mean() * 100)})
        print(f"\n{annee} – {type_bien}")
        print(resume.round(1).to_string())

# Les 10 sous-bassins où H7v3 perd le plus contre la naïve : appartements, 2024 (t contient la dernière fenêtre)
print("\nAppartements 2024 : les 10 sous-bassins où H7v3 perd le plus contre la naïve")
pire = t[t["type_bien"] == "appartement"].nlargest(10, "écart (€/m²)")
print(pire[["sous_bassin_de_vie", "BV2022", "ventes", "prix N-1 (€/m²)", "modele", "naive", "écart (€/m²)"]]
      .rename(columns={"modele": "MAE modèle (€/m²)", "naive": "MAE naïve (€/m²)"}).round(0).to_string(index=False))


2023 – appartement
                  sous-bassins  prix N-1 médian (€/m²)  MAE modèle (€/m²)  MAE naïve (€/m²)  écart moyen (€/m²)  sous-bassins où le modèle gagne (%)
prix N-1 (€/m²)                                                                                                                                     
Q1 (moins chers)           212                  1292.0              367.8             385.5               -17.7                                 63.2
Q2                         212                  1953.5              528.0             548.2               -20.2                                 64.6
Q3                         212                  2731.0              640.8             687.6               -46.8                                 67.0
Q4 (plus chers)            212                  4199.5             1302.2            1309.2                -7.0                                 56.6

2023 – maison
                  sous-bassins  prix N-1 médian (€/m²)  MAE modèle (€/m

**Résultats :**
- H7v3 bat la naïve dans tous les quartiles de prix pour les maisons, les deux années (écart de −22 à −77 €/m², 67 à 74 % des sous-bassins), et pour les appartements en 2023.
- Pour les appartements en 2024, il bat la naïve dans les trois premiers quartiles, mais perd dans le quartile le plus cher (écart moyen de +111 €/m²). Les dix sous-bassins où il perd le plus sont neuf arrondissements centraux de Paris (du 1er au 8e et le 16e), où le prix N-1 dépasse 11 000 €/m², et un sous-bassin de Savoie (73304). L'erreur par vente y atteint 3 700 à 6 600 €/m², contre 2 500 à 3 900 €/m² pour la naïve. C'est ce petit groupe qui rend l'écart moyen positif, alors que le modèle fait mieux dans la plupart des sous-bassins (section 5).
- Hypothèse : ces niveaux de prix sont rares dans l'apprentissage, et le réseau ramène les prévisions extrêmes vers le milieu de la distribution.

### 6.2 Importance des blocs de variables (H7v3)
Importance par permutation, sur l'année jugée, par type de bien : on mélange au hasard les lignes d'un bloc de variables entre les ventes (toutes les colonnes du bloc ensemble, par exemple les cinq indicatrices du nombre de pièces), 5 fois, et on mesure de combien l'erreur moyenne par vente augmente (pondérée par `w_h`). Une hausse nette : le modèle s'appuie sur ce bloc ; proche de 0 : il ne s'en sert pas. Une hausse se lit avec son écart-type : un bloc compte si elle le dépasse nettement.

Limite : deux blocs qui portent la même information (le sous-bassin et le prix N-1 disent tous deux le niveau des prix du lieu) se partagent l'importance ; mélanger l'un laisse l'autre en place, et l'importance de chacun est sous-estimée.

In [61]:
BLOCS = {"sous-bassin (embedding)": ["sous_bassin_de_vie"], "surface": ["log_sbati"], "terrain": ["log1p_sterr"],
         "pièces": COLONNES_PIECES, "dépendances": COLONNES_DEP, "territoire": BLOC_TERRITOIRE,
         "prix N-1": ["log_prix_n1"]}
N_REPETITIONS = 5
rng = np.random.default_rng(commun.GRAINE)

importances = []
for annee, (app, ev) in FENETRES.items():
    m = ajustes[(H7V3, annee)]
    for type_bien, e in ev.groupby("type_bien"):
        def mae(X):
            return np.average((e["prix_m2"] - m.predict(X)).abs(), weights=e["w_h"])
        base = mae(e)
        for bloc, colonnes in BLOCS.items():
            hausses = []
            for _ in range(N_REPETITIONS):
                X = e.copy()
                X[colonnes] = e[colonnes].to_numpy()[rng.permutation(len(e))]   # mêmes lignes mélangées pour tout le bloc
                hausses.append(mae(X) - base)
            importances.append({"année jugée": annee, "type de bien": type_bien, "bloc": bloc,
                                "hausse de la MAE (€/m²)": np.mean(hausses), "écart-type (€/m²)": np.std(hausses)})

importances = pd.DataFrame(importances)
importances.pivot_table(index="bloc", columns=["type de bien", "année jugée"],
                        values=["hausse de la MAE (€/m²)", "écart-type (€/m²)"]).round(1)

hausse de la MAE (€/m²)                      écart-type (€/m²)                 
type de bien                        appartement        maison              appartement      maison     
année jugée                                2023   2024   2023   2024              2023 2024   2023 2024
bloc                                                                                                   
dépendances                                16.2   11.3    9.1    9.3               0.6  0.7    0.6  0.3
pièces                                     42.1   21.7   13.1   11.3               1.8  0.9    0.5  0.5
prix N-1                                 1058.3  806.1  686.9  569.6               5.6  1.6    1.1  4.5
sous-bassin (embedding)                    37.0   54.2   20.5   19.5               1.4  1.5    0.6  0.3
surface                                   193.3  153.3   91.1   85.8               2.2  3.1    1.0  0.6
terrain                                     0.0    0.0   84.0   83.6               0.0  0.0    1.2  1.3
territoire                                114.5  140.9   46.5   67.9               0.5  1.5    1.0  1.1

**Comment lire :** prenons la surface, appartements, 2023 : la hausse vaut 193,3 €/m² avec un écart-type de 2,2. Quand on mélange les surfaces entre les ventes, l'erreur moyenne par vente augmente de 193 €/m² ; la hausse dépasse très nettement l'écart-type : le modèle s'appuie bien sur la surface.

**Résultats :**
- Le prix N-1 porte l'essentiel de la prévision : le mélanger fait monter l'erreur de 570 à 1 060 €/m² selon l'année et le type de bien.
- Le bien vient ensuite. La surface est le bloc le plus utile après le prix N-1 (86 à 193 €/m²) ; pour les maisons, le terrain compte autant que la surface (84 €/m²), ce qu'annonçait le constat (section 2.2). Le nombre de pièces (11 à 42 €/m²) et les dépendances (9 à 16 €/m²) apportent peu à côté de la surface.
- Le territoire (47 à 141 €/m²) compte plus que le sous-bassin (20 à 54 €/m²) : les deux portent la même information sur le lieu, et le réseau s'appuie surtout sur le territoire. C'est cohérent avec H7v2b, où le territoire remplaçait presque le sous-bassin.
- Le prix N-1 dit lui aussi le niveau des prix du lieu : l'importance du sous-bassin et du territoire est donc sous-estimée (limite de la méthode).

## 7. Conclusion

**Question :** un modèle par vente sépare-t-il l'effet du bien et celui du marché, et fait-il mieux que les modèles agrégés à information égale ? Avec les indicateurs du niveau 2 (volume et élan, capacité d'achat, trimestres précédents), anticipe-t-il mieux le retournement ?

**1. Le lieu et le bien ne suffisent pas.** Sans information sur le moment (H7v1, H7v1b, H7v2a, H7v2b et l'Elastic Net du « où »), les modèles font moins bien que la naïve, pour chaque vente (évaluation B) comme pour la médiane d'un sous-bassin (évaluation C). Ils sous-estiment les prix de 6 à 13 % : ils apprennent le niveau moyen de la période 2017 à l'année d'avant, alors que les prix ont fortement monté sur cette période. Les coordonnées de la parcelle n'apportent presque rien au sous-bassin (moins de 1 %) ; les variables de territoire du niveau 1 (parc, vacance, densité) décrivent presque aussi bien le lieu que l'identité du sous-bassin, sans la remplacer complètement.

**2. Le prix N-1 change tout.** C'est la seule comparaison retenue dans tous les cas : ajouté à H7v2a, il réduit l'erreur de 7 à 13 % par vente et de 42 à 57 % sur la médiane d'un sous-bassin, les deux années et pour les deux types de bien. Avec lui, le réseau bat la naïve pour prévoir le prix d'une maison vendue, les deux années (gain de 5,5 et 6,2 %) ; pour les appartements, seulement en 2023 (en 2024, il perd dans les arrondissements centraux de Paris, où les prix dépassent 11 000 €/m²).

**3. Le modèle sépare bien l'effet du bien.** Après le prix N-1, la surface est la variable la plus utile, et le terrain pour les maisons ; le nombre de pièces et les dépendances apportent peu à côté de la surface. C'est ce qui permet au modèle de battre la naïve vente par vente : la naïve donne le même prix à toutes les ventes d'un sous-bassin.

**4. Réseau ou Elastic Net : chacun son usage.** À informations égales, le réseau prévoit mieux le prix de chaque maison vendue (retenu contre l'Elastic Net, gain de 3,2 et 3,6 %). Pour la médiane d'un sous-bassin, l'Elastic Net fait nettement mieux, et c'est le seul modèle de H7 qui bat la naïve une année en évaluation C (maisons, 2024 : +6,5 %), sans être retenu (une année sur deux).

**5. Les indicateurs du « quand » n'apportent rien au réseau.** Le volume et l'élan, la capacité d'achat, les trimestres précédents, séparés ou réunis, ne réussissent aucune année contre H7v3. Ils déplacent le niveau des prévisions dans le bon sens pour les maisons en 2024 (le biais passe de +5,6 % à 0,0 % avec le taux d'effort, à +2,0 % avec les trimestres), sans réduire l'erreur. Le signal de fin 2023 était pourtant net (volume en baisse de 20 à 27 %, prix du 4e trimestre déjà en baisse) ; celui de fin 2022 était faible. Le réseau n'a jamais vu de retournement à l'apprentissage : il ne peut pas apprendre à l'anticiper.

**6. Il ne prévoit pas mieux le marché.** Pour la médiane d'un sous-bassin (évaluation C, la comparaison avec les modèles agrégés), l'effet du bien disparaît, et aucun réseau ne bat la naïve. La comparaison chiffrée avec les modèles agrégés (H4 à H6) est faite dans le rapport.

**Limites :**
- Le bootstrap mesure le hasard dû aux territoires, pas celui de l'entraînement du réseau : les petits écarts entre deux réseaux (quelques €/m²) peuvent dépendre de la graine.
- Les modèles apprennent sur l'échantillon de ventes (environ 10 % des ventes par département et par année, pondéré par `w_h`), pas sur toutes les ventes.
- Les variables de territoire sont des photos fixes, prises à un seul millésime pour toute la période.
- Un seul retournement dans les données (2023-2024) : aucun modèle ne peut apprendre à l'anticiper à partir d'un seul exemple.
- Les trimestres sont calculés au département, avec la valeur nationale pour 3,1 à 4,4 % des ventes ; l'élan inconnu est mis à 0 (0,3 % des ventes d'appartements au plus).